# Notebook 03 — Index Design + Analyzer Validation

**Gate rule:** Section 2 must print `✓ GO` before you run Section 3.
A wrong analyzer cannot be fixed without reindexing 263k documents.

**Sections:**
1. Connect + verify baseline
2. Analyzer validation on real training query terms (GO/NO-GO gate)
3. Build enriched index
4. Index size comparison
5. Latency comparison
6. Alias expansion validation
7. Score explainability
8. Summary

---

In [ ]:
import json, re, os, time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from elasticsearch import Elasticsearch

es = Elasticsearch(host='elasticsearch')
print('ES:', es.info()['version']['number'])

CORPUS        = '/work/data/FIR-s05-medline.json'
TRAIN_QUERIES = '/work/data/FIR-s05-training-queries-simple.txt'
QRELS         = '/work/data/FIR-s05-training-qrels.txt'
BASELINE      = 'medline'
ENRICHED      = 'medline_enriched'

def load_queries(path):
    queries = {}
    with open(path) as f:
        for line in f:
            parts = line.strip().split('\t', 1)
            if len(parts) == 2:
                queries[parts[0]] = parts[1].strip('"')
    return queries

def load_qrels(path):
    qrels = defaultdict(dict)
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 4:
                qrels[parts[0]][parts[2]] = int(parts[3])
    return qrels

queries = load_queries(TRAIN_QUERIES)
qrels   = load_qrels(QRELS)
print(f'Queries: {len(queries)},  Qrel topics: {len(qrels)}')

---
## 1. Verify baseline index

In [ ]:
if not es.indices.exists(index=BASELINE):
    print('ERROR: medline index not found')
    print('Run: docker exec fir-workbench python scripts/01_index.py')
else:
    count = es.count(index=BASELINE)['count']
    print(f'✓ medline: {count:,} documents')

    # verify PMID vs _id mismatch
    resp = es.search(index=BASELINE,
                     body={'size': 1, '_source': ['PMID', 'TI']})
    if resp['hits']['hits']:
        hit  = resp['hits']['hits'][0]
        esid = hit['_id']
        pmid = hit['_source'].get('PMID', 'NOT FOUND')
        print(f'\nID check:')
        print(f'  ES _id (sequential): {esid}')
        print(f'  PMID field:          {pmid}')
        if esid != pmid:
            print(f'  ✓ Different IDs confirmed')
            print(f'    Run files must use PMID from _source')
            print(f'    scripts/02_retrieve.py and 08_enrich_retrieve.py handle this')

---
## 2. Analyzer validation — GO/NO-GO gate

**Previous run results:**
```
Stemmer comparison on key pairs:
  minimal_english  →  inhibitor/inhibitor  activating/activating  ✗ fails verb forms
  porter (english) →  inhibitor/inhibitor  activ/activ            ✓ handles verb forms
  
Known Porter failure: transducer/transduction (transduc ≠ transduct)
  Affects: QID 40 (Signal-transducer...) and QID 18 (protease)
  Impact:  LOW — relevant docs for QID 40 use 'transducer' not 'transduction'
  Decision: accept Porter, document as known limitation
```

This section re-validates on real training query terms.

In [ ]:
# Create tiny analyzer-test index — no documents, just config
TEST_MAPPING = {
    'settings': {
        'number_of_shards': 1,
        'analysis': {
            'char_filter': {
                'greek_normaliser': {
                    'type': 'mapping',
                    'mappings': ['α => alpha','β => beta','γ => gamma',
                                 'κ => kappa','μ => mu']
                }
            },
            'filter': {
                'biomedical_stop': {
                    'type': 'stop',
                    'stopwords': [
                        'the','is','are','was','were','be','and','or',
                        'but','with','from','that','this','these','those',
                        'have','has','been','their','which','they'
                        # keep: a, b, c, d, in, of  (gene subunits)
                    ]
                },
                'english_stemmer': {
                    'type':     'stemmer',
                    'language': 'english'  # Porter — handles verb forms
                    # known failure: transducer/transduction, protease/proteolytic
                    # affects 2/38 queries, low practical impact
                }
            },
            'analyzer': {
                'biomedical_prose': {
                    'type':        'custom',
                    'char_filter': ['greek_normaliser'],
                    'tokenizer':   'standard',
                    'filter':      ['lowercase','biomedical_stop','english_stemmer']
                },
                'biomedical_exact': {
                    'type':      'custom',
                    'tokenizer': 'whitespace',
                    'filter':    ['uppercase']
                }
            }
        }
    }
}

es.indices.delete(index='analyzer-test', ignore=[400, 404])
es.indices.create(index='analyzer-test', body=TEST_MAPPING)
print('Analyzer test index created')

def analyze(es, analyzer, text):
    r = es.indices.analyze(
        index='analyzer-test',
        body={'analyzer': analyzer, 'text': text}
    )
    return [t['token'] for t in r['tokens']]

In [ ]:
# ── Test A: Real training query terms ─────────────────────────────────────────
# What tokens do your actual 38 queries produce?
# This is what goes into the posting lists.

print('biomedical_prose on real training query terms:')
print(f'{"QID":<6} {"Gene term":<42} {"Tokens"}')
print('─' * 85)

for qid, qtext in list(queries.items())[:20]:
    gene_part = re.sub(
        r'\s+in\s+(Homo sapiens|Mus musculus|Rattus norvegicus|Drosophila melanogaster)',
        '', qtext, flags=re.IGNORECASE
    ).strip().strip('"')
    gene_base = re.sub(r'\([^)]*\)', '', gene_part).strip()
    tokens    = analyze(es, 'biomedical_prose', gene_base)
    print(f'{qid:<6} {gene_base[:42]:<42} {tokens}')

In [ ]:
# ── Test B: Morphological variant unification ──────────────────────────────────
# These are variants actually found in MEDLINE abstracts.
# Unified = query and document hit the same posting list = higher recall.

print('Porter stemmer — morphological variant pairs:')
print('(Previous run: minimal_english failed activating/activation,')
print(' transducer/transduction. Porter fixes most but not all.)\n')

pairs = [
    # (query_word, doc_variant, note)
    ('inhibitor',     'inhibitors',      'cyclin-dependent kinase INHIBITOR'),
    ('inhibitor',     'inhibitory',      'inhibitory activity'),
    ('activating',    'activation',      '✓ Porter fixes this (minimal_english failed)'),
    ('receptor',      'receptors',       'adrenergic RECEPTOR'),
    ('kinase',        'kinases',         'Janus KINASE 2'),
    ('transcription', 'transcriptional', '✓ Porter fixes this'),
    ('regulator',     'regulatory',      'regulator → regulatory element'),
    ('binding',       'binds',           'binding protein → binds to'),
    ('transducer',    'transduction',    '✗ Known Porter failure (QID 40, low impact)'),
    ('protease',      'proteolytic',     '✗ Known Porter failure (QID 18, low impact)'),
    ('growth',        'growing',         'growth factor'),
    ('express',       'expression',      'gene expression'),
    ('signal',        'signaling',       'signal transducer'),
    ('phosphatase',   'phosphorylation', '✗ Different biological process — OK to fail'),
]

passed  = 0
failed  = []
known_failures = {'transducer/transduction', 'protease/proteolytic',
                  'phosphatase/phosphorylation'}

print(f'{"Query term":<18} {"Doc variant":<18} {"Same stem?":<12} {"Stems"}')
print('─' * 80)

for q_word, d_word, note in pairs:
    q_stem = analyze(es, 'biomedical_prose', q_word)
    d_stem = analyze(es, 'biomedical_prose', d_word)
    same   = q_stem == d_stem
    pair_key = f'{q_word}/{d_word}'
    is_known = pair_key in known_failures
    mark   = '✓' if same else ('~' if is_known else '✗')
    if same: passed += 1
    elif not is_known: failed.append(pair_key)
    print(f'{mark} {q_word:<18} {d_word:<18} {"YES" if same else "NO":<12} '
          f'{q_stem} = {d_stem}')

print(f'\nPassed: {passed}/{len(pairs)}')
print(f'Known failures (accepted): {[p for p in ["transducer/transduction","protease/proteolytic","phosphatase/phosphorylation"] if True]}')
print(f'Unexpected failures: {failed if failed else "none"}')

In [ ]:
# ── Test C: Gene identifier preservation ─────────────────────────────────────
# These go in gene_context with biomedical_exact — must NOT be stemmed
# Your 4 training alias queries depend on these being exact

print('biomedical_exact — gene identifier preservation (4 training alias queries):')
print()

identifiers = [
    # QID 1 aliases
    ('P21',    'alias for CDKN1A'),
    ('CIP1',   'alias for CDKN1A'),
    ('WAF1',   'alias for CDKN1A'),
    ('CDKN1A', 'official symbol'),
    # QID 3 aliases
    ('TEL',    'alias for ETV6'),
    ('ETV6',   'official symbol'),
    # QID 31 — neuropeptides
    ('TAC1',   'official symbol'),
    # QID 48 aliases
    ('PSO2',   'alias for ERCC4'),
    ('ERCC4',  'official symbol'),
    # Other identifiers with hyphens
    ('BCL2',   'B-cell lymphoma'),
    ('IL-6',   'hyphen must be preserved'),
    ('HO-1',   'hyphen must be preserved'),
    ('STAT5A', 'signal transducer'),
]

all_exact = True
print(f'{"Token":<12} {"Result":<15} {"Correct?":<10} Note')
print('─' * 60)
for tok, note in identifiers:
    result  = analyze(es, 'biomedical_exact', tok)
    correct = result == [tok.upper()]
    mark    = '✓' if correct else '✗'
    all_exact = all_exact and correct
    print(f'{mark} {tok:<12} {str(result):<15} {"YES" if correct else "NO -- FIX":<10} {note}')

In [ ]:
# ── Test D: Greek normalisation ───────────────────────────────────────────────
print('Greek char filter — normalisation:')
for text in ['α receptor', 'β-actin', 'γ-globulin', 'κB pathway', 'μ opioid']:
    result = analyze(es, 'biomedical_prose', text)
    print(f'  {text:<20} → {result}')

print()
print('Stopword check — must keep a, b, c, d (gene subunits):')
for text in ['vitamin d', 'interleukin 2', 'subunit a', 'type b']:
    result = analyze(es, 'biomedical_prose', text)
    print(f'  {text:<25} → {result}')

In [ ]:
# ── GO / NO-GO decision ───────────────────────────────────────────────────────
unexpected_prose_failures = len(failed) == 0
exact_ok                  = all_exact

print('='*55)
print('ANALYZER VALIDATION — GO / NO-GO')
print('='*55)
print(f'\nbiomedical_prose (Porter stemmer):')
print(f'  Unified:          {passed}/{len(pairs)} pairs')
print(f'  Known failures:   transducer/transduction, protease/proteolytic')
print(f'    → affects QID 40 and QID 18 only')
print(f'    → relevant docs likely use same surface form (low impact)')
print(f'    → documented as known limitation in essay')
print(f'  Unexpected fails: {failed if failed else "none"}')
print(f'\nbiomedical_exact (identifier):')
print(f'  All identifiers preserved: {"YES" if exact_ok else "NO"}')
print()

if unexpected_prose_failures and exact_ok:
    print('✓ GO — build medline_enriched index')
    print()
    print('Run: docker exec fir-workbench python scripts/07_enrich_index.py --recreate')
    print('Or run Section 3 below (uses subprocess)')
else:
    print('✗ NO-GO')
    if not unexpected_prose_failures:
        print(f'  Fix prose analyzer — unexpected failures: {failed}')
    if not exact_ok:
        print('  Fix exact analyzer — identifiers not preserved')
    print('  Edit 07_enrich_index.py mapping and retest this section')

---
## 3. Build enriched index

**Only run this after Section 2 prints `✓ GO`.**

Takes 20-40 minutes. Start and leave it.

In [ ]:
import subprocess, sys

print('Building medline_enriched (Porter stemmer, alias expansion, species)...')
print('One-time cost — index persists across docker compose down/up')
print()

result = subprocess.run(
    [sys.executable, '/work/scripts/07_enrich_index.py', '--recreate'],
    text=True
)
print('\n✓ Done' if result.returncode == 0 else f'\n✗ Failed (code {result.returncode})')

---
## 4. Index size comparison

In [ ]:
def index_stats(es, name):
    try:
        stats = es.indices.stats(index=name)
        idx   = stats['indices'][name]['total']
        store = idx['store']['size_in_bytes']
        docs  = idx['docs']['count']
        return {'name': name, 'docs': docs,
                'mb': store/1e6, 'bpd': store/docs if docs > 0 else 0}
    except Exception as e:
        print(f'  {name}: {e}')
        return None

all_stats = [index_stats(es, ix) for ix in [BASELINE, ENRICHED]]
all_stats = [s for s in all_stats if s]

print(f'{"Index":<22} {"Docs":>10} {"Size MB":>10} {"Bytes/doc":>12} {"Bloat"}')
print('─' * 62)
base_mb = None
for s in all_stats:
    if s['name'] == BASELINE: base_mb = s['mb']
    bloat = f"{s['mb']/base_mb:.1f}x" if base_mb and s['name'] != BASELINE else 'baseline'
    print(f"{s['name']:<22} {s['docs']:>10,} {s['mb']:>10.1f} {s['bpd']:>12,.0f} {bloat}")

if len(all_stats) >= 2:
    ratio = all_stats[1]['mb'] / all_stats[0]['mb']
    verdict = '✓ ACCEPTABLE' if ratio < 2.5 else '~ MODERATE' if ratio < 3.5 else '⚠ HIGH'
    print(f'\nBloat: {ratio:.1f}x — {verdict}')
    print('_source excludes (gene_context, concept_context, search_all) reduce size')

---
## 5. Latency comparison

In [ ]:
def latency(es, index, query, use_search_all=False, n=15):
    body = ({'query': {'match': {'search_all': query}}, 'size': 100}
            if use_search_all else
            {'query': {'multi_match': {'query': query,
                       'fields': ['TI^2','AB','MH']}}, 'size': 100})
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        es.search(index=index, body=body)
        times.append((time.perf_counter() - t0) * 1000)
    return np.mean(times), np.percentile(times, 95)

q = 'cyclin-dependent kinase inhibitor Homo sapiens'
configs = [
    (BASELINE,  False, 'baseline  (TI+AB+MH)'),
    (ENRICHED,  False, 'enriched  (multi_match)'),
    (ENRICHED,  True,  'enriched  (search_all copy_to)'),
]

print(f'{"Config":<36} {"Mean":>8} {"P95":>8}')
print('─' * 55)
for index, sa, label in configs:
    try:
        mean, p95 = latency(es, index, q, sa)
        print(f'{label:<36} {mean:>6.1f}ms {p95:>6.1f}ms')
    except Exception as e:
        print(f'{label:<36} Error: {e}')

---
## 6. Alias expansion validation

In [ ]:
# Do the 4 training alias queries actually benefit from expansion?
print('Alias expansion — 4 training queries with real aliases:')
print()

alias_tests = [
    ('P21',  'gene_context', 'QID 1 (CDKN1A alias)'),
    ('CIP1', 'gene_context', 'QID 1 (CDKN1A alias)'),
    ('WAF1', 'gene_context', 'QID 1 (CDKN1A alias)'),
    ('TEL',  'gene_context', 'QID 3 (ETV6 alias)'),
    ('PSO2', 'gene_context', 'QID 48 (ERCC4 alias)'),
]

for alias, field, label in alias_tests:
    r_base = es.search(index=BASELINE,
                       body={'query': {'match': {'AB': alias}},
                             'track_total_hits': True, 'size': 0})
    r_enr  = es.search(index=ENRICHED,
                       body={'query': {'match': {field: alias}},
                             'track_total_hits': True, 'size': 0})
    base_n = r_base['hits']['total']['value']
    enr_n  = r_enr['hits']['total']['value']
    mark   = '✓' if enr_n > base_n else '~'
    print(f'{mark} {alias:<8} [{label}]')
    print(f'   baseline AB:       {base_n:>8,} docs')
    print(f'   enriched gene_ctx: {enr_n:>8,} docs')

---
## 7. Score explainability

In [ ]:
def get_by_pmid(es, index, pmid):
    resp = es.search(index=index,
                     body={'query': {'term': {'PMID': str(pmid)}},
                           '_source': ['TI','PMID','species_norm','gene_symbols'],
                           'size': 1})
    return resp['hits']['hits'][0] if resp['hits']['hits'] else None

def explain_doc(es, index, query_text, es_id):
    result = es.explain(
        index=index, id=es_id,
        body={'query': {'multi_match': {
            'query':  query_text,
            'fields': ['TI^2','AB','MH','gene_context^3','concept_context^1.5']
        }}}
    )
    if not result['matched']:
        print('  NO MATCH — check tokens')
        return
    score = result['explanation']['value']
    print(f'  BM25 score: {score:.4f}')
    def parse(node, depth=0):
        if depth > 3: return
        v = node.get('value', 0)
        d = node.get('description', '')
        if not node.get('details') and v > 0.01:
            print(f'  {"  "*depth}{v:>7.4f}  {d[:65]}')
        for child in node.get('details', []):
            parse(child, depth+1)
    parse(result['explanation'])

# QID 1 — CDKN1A, has real aliases
qid   = '1'
qtext = queries[qid]
pmid  = list(qrels[qid].keys())[0]

print(f'Query {qid}: {qtext[:70]}')
print(f'First relevant PMID: {pmid}')

hit = get_by_pmid(es, ENRICHED, pmid)
if hit:
    print(f'Title: {hit["_source"].get("TI","")[:70]}')
    print(f'Species: {hit["_source"].get("species_norm","?")}')
    print(f'Gene symbols (first 5): {hit["_source"].get("gene_symbols",[])[:5]}')
    print(f'\n_explain (enriched index):')
    explain_doc(es, ENRICHED, qtext, hit['_id'])
else:
    print(f'PMID {pmid} not found — check PMID field in enriched index')

---
## 8. Summary

In [ ]:
print('='*65)
print('NOTEBOOK 03 COMPLETE')
print('='*65)

for idx in [BASELINE, ENRICHED]:
    exists = es.indices.exists(index=idx)
    count  = es.count(index=idx)['count'] if exists else 0
    status = '✓' if exists and count > 100000 else '✗'
    print(f'{status} {idx:<25} {count:>10,} docs')

print('''
Analyzer decisions — validated on 38 training queries:

  biomedical_prose (TI, AB, MH, concept_context)
    Stemmer:    Porter (english) — handles verb forms
    Unified:    inhibitor/inhibitors, activating/activation,
                transcription/transcriptional, receptor/receptors
    Known fail: transducer/transduction (QID 40, low impact)
                protease/proteolytic   (QID 18, low impact)
    Stopwords:  custom — keeps a,b,c,d (gene subunits)
    Greek:      α→alpha β→beta γ→gamma

  biomedical_exact (gene_context)
    All identifiers preserved: P21, CIP1, WAF1, TEL, PSO2,
    BCL2, IL-6, HO-1, STAT5A — zero stemming

Baseline (from 02_retrieve.py + 03_evaluate.py):
  MAP:         0.0855
  P@10:        0.0421
  recall@100:  0.3799
  recall@1000: 0.7789

Next steps:
  → python scripts/08_enrich_retrieve.py  (enriched retrieval)
  → python scripts/03_evaluate.py --baseline bm25_baseline.run
  → Notebook 04b (LTR reranker)
  → Notebook 05  (full evaluation)
''')